<a href="https://colab.research.google.com/github/Lalepragati/Behavior-/blob/main/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import os

# Check if the mount point exists
if os.path.exists('/content/drive'):
    print("Google Drive mount point exists.")
    # Optionally, list the contents of the drive to confirm accessibility
    print("Contents of /content/drive/MyDrive:")
    try:
        for item in os.listdir('/content/drive/MyDrive'):
            print(f"- {item}")
    except Exception as e:
        print(f"Could not list contents: {e}")
else:
    print("Google Drive is NOT mounted.")

Google Drive mount point exists.
Contents of /content/drive/MyDrive:
- colab_models


In [39]:
MODEL_DIR = Path('/content/drive/MyDrive/colab_models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / 'isolation_forest.pkl'
pickle.dump(model, open(MODEL_PATH, 'wb'))

In [41]:
# Save the training_frame to Google Drive as a CSV file
training_frame_path = MODEL_DIR / 'training_frame.csv'
training_frame.to_csv(training_frame_path, index=False)
print(f"Saved training_frame to {training_frame_path.resolve()}")

Saved training_frame to /content/drive/MyDrive/colab_models/training_frame.csv


In [42]:
training_frame_path = MODEL_DIR / 'training_frame.csv'
if training_frame_path.exists():
    print(f"'training_frame.csv' exists at: {training_frame_path}")
else:
    print(f"'training_frame.csv' does NOT exist at: {training_frame_path}")

'training_frame.csv' exists at: /content/drive/MyDrive/colab_models/training_frame.csv


In [43]:
# Core imports for feature engineering and model training.
import json
import math
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

PASS_PHRASE = "mysecurepassword"
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / "isolation_forest.pkl"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook ready.")

Notebook ready.


In [44]:
# Replace this inline dataset with your own keystroke timing exports if needed.
sample_data = [
    {
        "username": "demo_user",
        "value": PASS_PHRASE,
        "characters": list(PASS_PHRASE),
        "down_times": [0, 142, 281, 423, 565, 701, 844, 986, 1121, 1264, 1407, 1549, 1690, 1834, 1978, 2119],
        "up_times":   [105, 235, 371, 504, 651, 781, 921, 1058, 1203, 1337, 1487, 1619, 1760, 1904, 2043, 2194],
    },
    {
        "username": "demo_user",
        "value": PASS_PHRASE,
        "characters": list(PASS_PHRASE),
        "down_times": [0, 138, 279, 419, 559, 703, 847, 991, 1129, 1270, 1411, 1556, 1699, 1840, 1981, 2126],
        "up_times":   [101, 229, 367, 502, 647, 782, 922, 1062, 1204, 1345, 1491, 1624, 1770, 1909, 2048, 2198],
    },
    {
        "username": "demo_user",
        "value": PASS_PHRASE,
        "characters": list(PASS_PHRASE),
        "down_times": [0, 145, 290, 435, 580, 725, 871, 1015, 1160, 1306, 1450, 1595, 1739, 1884, 2028, 2174],
        "up_times":   [108, 240, 381, 522, 661, 804, 948, 1090, 1236, 1377, 1520, 1664, 1808, 1950, 2097, 2239],
    },
]

print(f"Loaded {len(sample_data)} training samples.")

Loaded 3 training samples.


In [45]:
# Replace this inline dataset with your own keystroke timing exports if needed.
sample_data = [
    {
        "username": "demo_user",
        "value": PASS_PHRASE,
        "characters": list(PASS_PHRASE),
        "down_times": [0, 142, 281, 423, 565, 701, 844, 986, 1121, 1264, 1407, 1549, 1690, 1834, 1978, 2119],
        "up_times":   [105, 235, 371, 504, 651, 781, 921, 1058, 1203, 1337, 1487, 1619, 1760, 1904, 2043, 2194],
    },
    {
        "username": "demo_user",
        "value": PASS_PHRASE,
        "characters": list(PASS_PHRASE),
        "down_times": [0, 138, 279, 419, 559, 703, 847, 991, 1129, 1270, 1411, 1556, 1699, 1840, 1981, 2126],
        "up_times":   [101, 229, 367, 502, 647, 782, 922, 1062, 1204, 1345, 1491, 1624, 1770, 1909, 2048, 2198],
    },
    {
        "username": "demo_user",
        "value": PASS_PHRASE,
        "characters": list(PASS_PHRASE),
        "down_times": [0, 145, 290, 435, 580, 725, 871, 1015, 1160, 1306, 1450, 1595, 1739, 1884, 2028, 2174],
        "up_times":   [108, 240, 381, 522, 661, 804, 948, 1090, 1236, 1377, 1520, 1664, 1808, 1950, 2097, 2239],
    },
]

print(f"Loaded {len(sample_data)} training samples.")

Loaded 3 training samples.


In [46]:
# Feature engineering mirrors the Flask app so the exported model matches runtime behavior.
def safe_std(values):
    return float(np.std(values, ddof=0)) if len(values) > 1 else 0.0

def extract_features(sample):
    value = str(sample.get("value", "")).strip()
    if value.lower() != PASS_PHRASE:
        raise ValueError("Passphrase mismatch.")

    characters = sample.get("characters") or []
    down_times = [float(x) for x in sample.get("down_times") or []]
    up_times = [float(x) for x in sample.get("up_times") or []]

    if len(characters) < 2:
        raise ValueError("At least two characters are required.")
    if len(characters) != len(down_times) or len(characters) != len(up_times):
        raise ValueError("Timing arrays must match the number of characters.")

    dwell_times = [max(0.0, up_time - down_time) for down_time, up_time in zip(down_times, up_times)]
    flight_times = [max(0.0, down_times[i] - up_times[i - 1]) for i in range(1, len(down_times))]
    session_duration = max(0.0, up_times[-1] - down_times[0])
    typing_speed = float(len(characters) / (session_duration / 1000.0)) if session_duration > 0 else 0.0

    return {
        "avg_dwell": float(np.mean(dwell_times)) if dwell_times else 0.0,
        "avg_flight": float(np.mean(flight_times)) if flight_times else 0.0,
        "std_dwell": safe_std(dwell_times),
        "std_flight": safe_std(flight_times),
        "typing_speed": typing_speed,
        "session_duration": session_duration,
        "feature_vector": [
            float(np.mean(dwell_times)) if dwell_times else 0.0,
            float(np.mean(flight_times)) if flight_times else 0.0,
            safe_std(dwell_times),
            safe_std(flight_times),
            typing_speed,
            session_duration,
        ],
    }

feature_rows = [extract_features(sample) for sample in sample_data]
training_frame = pd.DataFrame([row["feature_vector"] for row in feature_rows], columns=["avg_dwell", "avg_flight", "std_dwell", "std_flight", "typing_speed", "session_duration"])
training_frame.head()

,avg_dwell,avg_flight,std_dwell,std_flight,typing_speed,session_duration
0,79.3125,61.666667,9.998242,10.848451,7.292616,2194.0
1,78.3125,63.000000,9.318790,10.417933,7.279345,2198.0
2,78.0000,66.066667,11.543396,11.310565,7.146047,2239.0


In [47]:
# Train the anomaly detector and persist the artifact as a pickle file.
model = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
)
model.fit(training_frame)

with MODEL_PATH.open("wb") as handle:
    pickle.dump(model, handle)

print(f"Saved model to {MODEL_PATH.resolve()}")

Saved model to /content/models/isolation_forest.pkl


In [48]:
# Colab-only download helper. Uncomment the last two lines when running in Google Colab.
try:
    from google.colab import files

    # files.download(str(MODEL_PATH))
    print("Google Colab detected. Uncomment files.download(...) to download the model.")
except Exception:
    print("Not running in Colab. Copy the saved pickle into the Flask app's models/ folder manually.")

Google Colab detected. Uncomment files.download(...) to download the model.
